# MLB Analytics Hub — XGBoost Production Artifact Generator

This notebook is a **thin Colab wrapper around the canonical training
pipeline** in `train_prop_models.py`. It does *not* re-implement feature
engineering — that previously caused the notebook to drift from
`xgb_prop_scorer.py` (mismatched feature names, placeholder targets, no
arsenal pitch-mix), producing models the scorer could not use.

Running this trains **all 7 markets** (hits, hr, tb, rbi, k_3.5/4.5/5.5)
with:
- real per-game targets aggregated from Statcast,
- the exact feature columns `xgb_prop_scorer._build_*_features` produces,
- usage-weighted **arsenal pitch-mix** features for the K models
  (`arsenal_whiff_pct`, `arsenal_putaway_pct`).

It writes `models/xgb_*.pkl` + `models/xgb_feature_cols.json` ready to
commit into production. **Keep `xgboost==3.2.0` pinned** (artifact format).


In [ ]:
# ── CELL 1: Install pinned dependencies ──────────────────────────────────────
# xgboost MUST be 3.2.0 — the scorer pins this to keep the artifact format
# stable (see docs/xgb_model_regeneration.md).
!pip -q install "xgboost==3.2.0" pybaseball scikit-learn joblib pandas numpy
print('✅ deps installed')


In [ ]:
# ── CELL 2: Make the repo's training modules importable ──────────────────────
# The notebook delegates to train_prop_models.py from this repo, so the repo
# must be on sys.path. Two ways:
#   (A) git clone (set GITHUB_TOKEN in Colab Secrets for the private repo), or
#   (B) upload train_prop_models.py via the Files panel and set REPO_DIR='/content'.
import os, sys, subprocess

REPO_URL = 'https://github.com/johnmsimo/mlb-analytics-hub.git'
REPO_DIR = '/content/mlb-analytics-hub'
TOKEN    = os.environ.get('GITHUB_TOKEN', '')  # optional; needed for private clone

if not os.path.isfile(os.path.join(REPO_DIR, 'train_prop_models.py')):
    try:
        url = REPO_URL.replace('https://', f'https://{TOKEN}@') if TOKEN else REPO_URL
        subprocess.run(['git', 'clone', '--depth', '1', url, REPO_DIR], check=True)
    except Exception as e:
        print(f'⚠️  git clone failed ({e}).')
        print('   Fallback: upload train_prop_models.py via the Files panel,')
        print("   then set REPO_DIR = '/content' and re-run this cell.")

sys.path.insert(0, REPO_DIR)
import train_prop_models as tpm
print('✅ using train_prop_models from', tpm.__file__)
print('   models will be written to', tpm._MODEL_DIR)


In [ ]:
# ── CELL 3: Configure the run ────────────────────────────────────────────────
# Markets: omit / None = all 7. Seasons: training window.
SEASONS = [2021, 2022, 2023, 2024, 2025]
MARKETS = None   # e.g. ['k_3.5', 'k_4.5', 'k_5.5'] to train only the K models

# Optional live progress print so the long Statcast fetch shows movement.
def _progress(phase, log_line=None):
    msg = log_line or phase
    if msg:
        print(msg, flush=True)

print('Markets:', MARKETS or 'ALL (7)', ' Seasons:', SEASONS)


In [ ]:
# ── CELL 4: Train (real targets + arsenal pitch-mix, written to models/) ─────
# This pulls Statcast in ~2-week windows (memory-bounded), aggregates real
# per-game outcomes, fetches Savant arsenal stats, builds the exact scorer
# feature columns, and trains + calibrates each market. Expect this to take a
# while on the first run (full Statcast pulls).
results = tpm.train_all(markets=MARKETS, seasons=SEASONS, progress_cb=_progress)
print('\n✅ Training complete')


In [ ]:
# ── CELL 5: Verify artifacts (feature wiring + arsenal presence) ─────────────
import glob, joblib
MODEL_DIR = tpm._MODEL_DIR
for p in sorted(glob.glob(os.path.join(MODEL_DIR, 'xgb_*.pkl'))):
    art   = joblib.load(p)
    meta  = art.get('meta', {})
    feats = art.get('features', [])
    flags = ''
    if any(f.startswith('arsenal_') for f in feats):
        flags = '  ↳ arsenal pitch-mix ✓'
    print(f"{os.path.basename(p):<28} AUC={meta.get('auc')}  Brier={meta.get('brier')}  "
          f"feats={len(feats)}{flags}")
print('\nfeature-col map →', os.path.join(MODEL_DIR, 'xgb_feature_cols.json'))


## Deploying the artifacts

1. Download every `models/xgb_*.pkl` **and** `models/xgb_feature_cols.json`
   (left Files panel → `mlb-analytics-hub/models/`, or zip and download).
2. Replace the files in the production repo's `models/` directory and commit.
3. The running app picks them up on the next restart
   (`xgb_prop_scorer._load_models()`); the scorer reads each model's own
   saved `features` list, so the arsenal columns wire in automatically.

Because this notebook delegates to `train_prop_models.py`, it can never
drift from the scorer again — that module is the single source of truth for
feature engineering and is the same code the in-app **MODEL TRAINING** panel
runs.
